<a href="https://colab.research.google.com/github/tijesu24/Plant-disease-Recognition-papers/blob/main/Plantseg%20wt%20starnet%20replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
weitianqi_plantseg_path = kagglehub.dataset_download('weitianqi/plantseg')

print('Data source import complete.')


Data source import complete.


In [2]:
# ==========================================
# 0. ENVIRONMENT SETUP & DEPENDENCIES
# ==========================================
!pip install -q timm albumentations einops

import os
import re
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cuda


# 1. DATASET & MULTI-CLASS MAPPING SETUP

In [3]:

# ==========================================
# 1. DATASET & MULTI-CLASS MAPPING SETUP
# ==========================================
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}

def clean_label(stem):
    label = re.sub(r"_(google|bing|baidu)", "", stem, flags=re.IGNORECASE)
    label = re.sub(r"_([a-zA-Z]+_[a-zA-Z]+)\s*\(\d+\)", "", label)
    label = re.sub(r"_([a-zA-Z]+)\s*\(\d+\)", "", label)
    label = re.sub(r"\s*\(\d+\)", "", label)
    label = re.sub(r"_\d+$", "", label)
    return label.lower().strip()

def build_label_to_idx(split_dirs):
    class_names = set()
    for img_dir, mask_dir in split_dirs.values():
        for folder in (img_dir, mask_dir):
            class_names.update(
                clean_label(path.stem)
                for path in Path(folder).glob("*.*")
                if path.suffix.lower() in IMAGE_SUFFIXES
            )
    if not class_names or "" in class_names:
        raise ValueError("Could not derive valid class names from the split filenames.")
    return {name: index for index, name in enumerate(sorted(class_names))}

class PlantSegReplicationDataset(Dataset):
    """Dataset for paired PlantSeg images and segmentation masks."""
    def __init__(self, img_dir, mask_dir, label_to_idx, num_seg_classes, max_samples=None, transform=None):
        self.img_dir = Path(img_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform
        self.num_seg_classes = num_seg_classes
        img_files = {f.stem: f for f in self.img_dir.glob("*.*") if f.suffix.lower() in IMAGE_SUFFIXES}
        mask_files = {f.stem: f for f in self.mask_dir.glob("*.*") if f.suffix.lower() in IMAGE_SUFFIXES}
        common_stems = sorted(set(img_files) & set(mask_files))
        if max_samples and len(common_stems) > max_samples:
            np.random.seed(42)
            common_stems = list(np.random.choice(common_stems, max_samples, replace=False))
        self.samples = [(img_files[stem], mask_files[stem]) for stem in common_stems]
        cleaned_labels = [clean_label(stem) for stem in common_stems]
        unknown_labels = sorted(set(cleaned_labels) - set(label_to_idx))
        if unknown_labels:
            raise ValueError(f"Labels missing from the shared class map: {unknown_labels}")
        self.label_to_idx = label_to_idx
        self.labels = [self.label_to_idx[name] for name in cleaned_labels]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        image = cv2.imread(str(img_path))
        if image is None:
            raise ValueError(f"Could not read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        raw_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if raw_mask is None:
            raise ValueError(f"Could not read segmentation mask: {mask_path}")

        unique_labels = np.unique(raw_mask)
        invalid_labels = unique_labels[(unique_labels < 0) | (unique_labels >= self.num_seg_classes)]
        if invalid_labels.size:
            raise ValueError(
                f"Mask labels must be in [0, {self.num_seg_classes - 1}], but "
                f"{mask_path} contains out-of-range values {invalid_labels.tolist()}. "
                f"All grayscale values in this mask: {unique_labels.tolist()}. "
                "Check whether the mask uses an unsupported ignore value, color encoding, "
                "or a label mapping different from the model's classes."
            )

        mask = raw_mask.astype(np.int64)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"].long()
        else:
            mask = torch.tensor(mask, dtype=torch.long)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, mask, label



# The   Heterogeneous Branch Attention Aggregation module. 
It is an important part of the code. 

In [4]:
class WindowAttention4x4(nn.Module):
    """Matches Figure 3: 4x4 Windowed Self-Attention -> Conv2d -> BN -> GELU + Residual"""
    def __init__(self, dim, num_heads=4, window_size=4):
        super().__init__()
        self.ws = window_size
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.proj = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(dim),
            nn.GELU()
        )

    def forward(self, x):
        B, C, H, W = x.shape
        # Pad if H or W is not divisible by window_size (4)
        pad_h = (self.ws - H % self.ws) % self.ws
        pad_w = (self.ws - W % self.ws) % self.ws
        x_pad = F.pad(x, (0, pad_w, 0, pad_h))
        _, _, Hp, Wp = x_pad.shape

        # Partition into 4x4 windows
        w = x_pad.view(B, C, Hp // self.ws, self.ws, Wp // self.ws, self.ws)
        w = w.permute(0, 2, 4, 3, 5, 1).contiguous().view(-1, self.ws * self.ws, C)

        attn_out, _ = self.attn(w, w, w)

        # Reverse window partition
        attn_out = attn_out.view(B, Hp // self.ws, Wp // self.ws, self.ws, self.ws, C)
        attn_out = attn_out.permute(0, 5, 1, 3, 2, 4).contiguous().view(B, C, Hp, Wp)
        attn_out = attn_out[:, :, :H, :W]

        return x + self.proj(attn_out)

class DSConvAtrous(nn.Module):
    """Depthwise Separable Convolution with dilation"""
    def __init__(self, in_ch, out_ch, dilation):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, 3, padding=dilation, dilation=dilation, groups=in_ch, bias=False),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU()
        )

    def forward(self, x):
        return self.net(x)

class HBAAModule(nn.Module):
    def __init__(self, channels):
        super().__init__()
        ch_branch = channels // 6
        self.branch_1x1 = nn.Conv2d(channels, ch_branch, 1)
        self.branch_window = nn.Sequential(
            nn.Conv2d(channels, ch_branch, 1),
            WindowAttention4x4(ch_branch)
        )
        self.branch_atrous6 = DSConvAtrous(channels, ch_branch, dilation=6)
        self.branch_atrous12 = DSConvAtrous(channels, ch_branch, dilation=12)
        self.branch_strip = nn.Sequential(
            nn.Conv2d(channels, ch_branch, 1),
            StripPooling(ch_branch)
        )
        self.branch_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, ch_branch, 1)
        )

        self.cbam = CBAM(channels)
        self.fusion_conv = nn.Conv2d(channels, channels, 1)
        self.residual_conv = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        # All 6 branches process x in parallel
        p1 = self.branch_1x1(x)
        p2 = self.branch_window(x)
        p3 = self.branch_atrous6(x)
        p4 = self.branch_atrous12(x)
        p5 = self.branch_strip(x)
        p6 = F.interpolate(self.branch_pool(x), size=(h, w), mode='bilinear', align_corners=False)

        concat = torch.cat([p1, p2, p3, p4, p5, p6], dim=1)
        out = self.fusion_conv(self.cbam(concat))
        return out + self.residual_conv(x)

# CARAFE++

Used for smart upsampling

In [5]:
class CARAFE(nn.Module):
    """Memory-efficient CARAFE++ upsampler (Figure 4)"""
    def __init__(self, channels, scale_factor=2, c_mid=64, k_enc=3, k_up=5):
        super().__init__()
        self.scale = scale_factor
        self.k_up = k_up
        self.compressor = nn.Conv2d(channels, c_mid, kernel_size=1)
        self.encoder = nn.Conv2d(
            c_mid, (scale_factor ** 2) * (k_up ** 2),
            kernel_size=k_enc, padding=k_enc // 2
        )

    def forward(self, x):
        B, C, H, W = x.shape

        # 1. Predict and normalize content-aware kernels
        kernels = self.encoder(self.compressor(x))          # [B, scale^2 * k_up^2, H, W]
        kernels = F.pixel_shuffle(kernels, self.scale)      # [B, k_up^2, H*scale, W*scale]
        kernels = F.softmax(kernels, dim=1)
        kernels = kernels.view(B, self.k_up ** 2, H, self.scale, W, self.scale)

        # 2. Extract k_up x k_up local patches without duplicating memory
        patches = F.unfold(x, kernel_size=self.k_up, padding=self.k_up // 2)
        patches = patches.view(B, C, self.k_up ** 2, H, W)

        # 3. Reassemble features into the higher-resolution grid
        out = torch.einsum('bckhw,bkhywx->bchywx', patches, kernels)
        return out.reshape(B, C, H * self.scale, W * self.scale)




# 2. ARCHITECTURE: HBAA MODULE & STAR-NET

In [6]:
# ==========================================
# 2. ARCHITECTURE: HBAA MODULE & STAR-NET
# ==========================================
class StripPooling(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.conv = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        h_str = self.pool_h(x)
        w_str = self.pool_w(x)
        out = h_str.expand_as(x) + w_str.expand_as(x)
        return torch.sigmoid(self.conv(out)) * x

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels)
        )
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        max_out = F.adaptive_max_pool2d(x, (1, 1))
        avg_out = F.adaptive_avg_pool2d(x, (1, 1))
        ca = torch.sigmoid(self.mlp(max_out) + self.mlp(avg_out)).unsqueeze(2).unsqueeze(3)
        x_ca = x * ca

        max_s = torch.max(x_ca, dim=1, keepdim=True)[0]
        avg_s = torch.mean(x_ca, dim=1, keepdim=True)

        sa = torch.sigmoid(self.spatial_conv(torch.cat([max_s, avg_s], dim=1)))
        return x_ca * sa



class STARNet(nn.Module):
    def __init__(self, num_seg_classes=117, img_size=512):
        super().__init__()
        # Swin Transformer backbone (pretrained ImageNet-1K)
        self.backbone = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=True,
            features_only=True,
            img_size=img_size
        )
        self.hbaa = HBAAModule(768)  # Stage 4 output channels (1/32 scale)

        self.shallow_conv = nn.Conv2d(96, 96, 1)  # Stage 1 output channels (1/4 scale)
        self.shallow_cbam = CBAM(96)

        # NEW: CARAFE++ x8 upsampler for the deep path (1/32 -> 1/4 scale)
        self.carafe_deep = CARAFE(channels=768, scale_factor=8)

        # UPDATED: Keep only the 3x3 fusion block at 1/4 scale
        self.decoder_conv = nn.Sequential(
            nn.Conv2d(768 + 96, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        # NEW: CARAFE++ x4 upsampler for the fused path (1/4 -> 1/1 full scale)
        self.carafe_final = CARAFE(channels=256, scale_factor=4)

        # NEW: Final 1x1 segmentation head at full resolution
        self.seg_head = nn.Conv2d(256, num_seg_classes, kernel_size=1)

    def forward(self, x):
        features = self.backbone(x)
        stage1_feat = features[0].permute(0, 3, 1, 2).contiguous()  # [B, 96, H/4, W/4]
        stage4_feat = features[3].permute(0, 3, 1, 2).contiguous()  # [B, 768, H/32, W/32]

        # Deep path: HBAA enhancement -> CARAFE++ x8 upsampling
        enhanced_deep = self.hbaa(stage4_feat)
        upsampled_deep = self.carafe_deep(enhanced_deep)            # [B, 768, H/4, W/4]

        # Shallow path: 1x1 conv -> CBAM
        shallow = self.shallow_cbam(self.shallow_conv(stage1_feat)) # [B, 96, H/4, W/4]

        # Feature fusion and 3x3 smoothing
        fused = torch.cat([upsampled_deep, shallow], dim=1)         # [B, 864, H/4, W/4]
        fused = self.decoder_conv(fused)                            # [B, 256, H/4, W/4]

        # Final upsampling: CARAFE++ x4 -> 1x1 classification head
        upsampled_final = self.carafe_final(fused)                  # [B, 256, H, W]
        out = self.seg_head(upsampled_final)                        # [B, num_seg_classes, H, W]
        return out




# 3. LOSS FUNCTION: DPW-LOSS (3-PHASE CURRICULUM)

In [7]:
# ==========================================
# 3. LOSS FUNCTION: DPW-LOSS (3-PHASE CURRICULUM)
# ==========================================
class GeneralizedDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets, num_classes=117):
        preds = F.softmax(preds.float(), dim=1) # Force float32 to prevent fp16 overflow
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).permute(0, 3, 1, 2).float()

        target_sum = torch.sum(targets_one_hot, dim=(0, 2, 3))
        present_mask = (target_sum > 0).float() # Only weight classes present in the batch

        weights = present_mask / (target_sum ** 2 + self.smooth)
        intersection = torch.sum(preds * targets_one_hot, dim=(0, 2, 3))
        union = torch.sum(preds + targets_one_hot, dim=(0, 2, 3))

        gdl = 1.0 - (2.0 * torch.sum(weights * intersection) + self.smooth) / (torch.sum(weights * union) + self.smooth)
        return gdl

class DPWLoss(nn.Module):
    def __init__(self, num_classes=117):
        super().__init__()
        self.num_classes = num_classes
        self.ce = nn.CrossEntropyLoss()
        self.gdl = GeneralizedDiceLoss()
        
        # State trackers for phase transitions
        self.phase = 1
        self.transition_start = None
        self.val_lce_history = []
        self.val_steps_history = []

    def focal_loss(self, inputs, targets, alpha=0.25, gamma=2.0):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        return (alpha * (1 - pt) ** gamma * ce_loss).mean()

    def update_validation(self, val_lce, val_miou, current_step):
        """
        Called after every validation loop to check feedback-driven triggers[cite: 7, 8].
        """
        self.val_lce_history.append(val_lce)
        self.val_steps_history.append(current_step)

        # Trigger 1: Phase 1 -> Phase 2 based on L_CE convergence
        if self.phase == 1 and self.transition_start is None:
            if len(self.val_lce_history) >= 2:
                # Retrieve L_CE from ~10,000 iterations ago[cite: 8]
                old_lce = None
                for hist_lce, hist_step in zip(reversed(self.val_lce_history), reversed(self.val_steps_history)):
                    if (current_step - hist_step) >= 10000:
                        old_lce = hist_lce
                        break
                
                # If 10k steps haven't elapsed, check against the oldest recorded value
                if old_lce is None:
                    old_lce = self.val_lce_history[0]

                # Calculate absolute rate of change[cite: 8]
                rate_of_change = abs(val_lce - old_lce) / (old_lce + 1e-8)
                
                if rate_of_change < 0.005:
                    self.transition_start = current_step
                    self.phase = 1.5  # 1.5 indicates interpolation towards Phase 2

        # Trigger 2: Phase 2 -> Phase 3 based on mIoU target[cite: 8]
        elif self.phase == 2 and self.transition_start is None:
            if val_miou > 0.39:
                self.transition_start = current_step
                self.phase = 2.5  # 2.5 indicates interpolation towards Phase 3

    def get_dynamic_weights(self, step):
        # Target weights for each phase: [alpha, beta, gamma][cite: 8]
        p1 = torch.tensor([1.0, 0.1, 0.1], device=device)
        p2 = torch.tensor([0.1, 0.1, 1.0], device=device)
        p3 = torch.tensor([0.1, 1.0, 0.5], device=device)

        # 1. Failsafe triggers: Force transition if step count is reached[cite: 7, 8]
        if self.phase == 1 and step >= 80000:
            self.transition_start = 80000
            self.phase = 1.5
        elif self.phase == 2 and step >= 240000:
            self.transition_start = 240000
            self.phase = 2.5

        # 2. Apply weights and 2,000-iteration linear interpolation[cite: 8]
        if self.phase == 1:
            return p1[0], p1[1], p1[2]
            
        elif self.phase == 1.5:
            progress = min(1.0, (step - self.transition_start) / 2000.0)
            if progress >= 1.0:
                self.phase = 2
                self.transition_start = None
                return p2[0], p2[1], p2[2]
            w = (1.0 - progress) * p1 + progress * p2
            return w[0], w[1], w[2]
            
        elif self.phase == 2:
            return p2[0], p2[1], p2[2]
            
        elif self.phase == 2.5:
            progress = min(1.0, (step - self.transition_start) / 2000.0)
            if progress >= 1.0:
                self.phase = 3
                self.transition_start = None
                return p3[0], p3[1], p3[2]
            w = (1.0 - progress) * p2 + progress * p3
            return w[0], w[1], w[2]
            
        else:
            return p3[0], p3[1], p3[2]

    def forward(self, inputs, targets, global_step):
        # Fetch dynamically adjusted weights based on the current step[cite: 8]
        alpha, beta, gamma = self.get_dynamic_weights(global_step)

        l_ce = self.ce(inputs, targets)
        l_focal = self.focal_loss(inputs, targets)
        l_gdl = self.gdl(inputs, targets, self.num_classes)

        return (alpha * l_ce) + (beta * l_gdl) + (gamma * l_focal)


# Training code

In [8]:
import os
import time
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Define the data augmentations matching the paper's specifications
train_transform = A.Compose([
    A.Resize(512, 512),
    A.RandomScale(scale_limit=(-0.5, 1.0), p=1.0), 
    A.PadIfNeeded(min_height=512, min_width=512, border_mode=0), # cv2.BORDER_CONSTANT=0
    A.RandomCrop(512, 512),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Validation transforms only resize and normalize
val_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

def run_replication_training(resume_checkpoint=None):
    # 1. Directory Setup 
    dataset_root = Path("/kaggle/input/datasets/weitianqi/plantseg/plantsegv2")
    images_root = dataset_root / "images"
    annotations_root = dataset_root / "annotations"
    split_dirs = {}
    for split_name in ("train", "val", "test"):
        img_dir = images_root / split_name
        mask_dir = annotations_root / split_name
        if img_dir.exists() != mask_dir.exists():
            raise FileNotFoundError(
                f"The {split_name} split must have both image and annotation folders: "
                f"images={img_dir}, annotations={mask_dir}"
            )
        if img_dir.is_dir() and mask_dir.is_dir():
            split_dirs[split_name] = (img_dir, mask_dir)

    for required_split in ("train", "val"):
        if required_split not in split_dirs:
            raise FileNotFoundError(
                f"Required {required_split} image and annotation folders were not found under {dataset_root}."
            )

    # Use one filename-derived class map for every available split.
    label_to_idx = build_label_to_idx(split_dirs)
    # Mask ID 116 requires at least 117 segmentation output channels.
    num_seg_classes = 117
    print(f"Found {len(label_to_idx)} filename classes across splits: {list(split_dirs)}")

    # 2. Datasets & DataLoaders
    train_dataset = PlantSegReplicationDataset(
        img_dir=split_dirs["train"][0],
        mask_dir=split_dirs["train"][1],
        label_to_idx=label_to_idx,
        num_seg_classes=num_seg_classes,
        transform=train_transform,
    )
    val_dataset = PlantSegReplicationDataset(
        img_dir=split_dirs["val"][0],
        mask_dir=split_dirs["val"][1],
        label_to_idx=label_to_idx,
        num_seg_classes=num_seg_classes,
        transform=val_transform,
    )

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

    # 3. Model, Loss, Optimizer Initialization
    model = STARNet(num_seg_classes=num_seg_classes).to(device)
    criterion = DPWLoss(num_classes=num_seg_classes).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=6e-5, betas=(0.9, 0.999), weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    # Implement Polynomial Decay to exact eta_min floor[cite: 9]
    total_iterations = 320000
    lr_init = 6e-5
    lr_min = 1e-6
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda step: (
            (lr_min + (lr_init - lr_min) * ((1 - min(step, total_iterations) / total_iterations) ** 0.9)) / lr_init
        )
    )

    global_step = 0
    start_epoch = 16
    epochs = 10 # Set to a low number for Kaggle 12-hour session blocks

    # 4. Checkpoint Resumption (Crucial for Kaggle)
    if resume_checkpoint and os.path.exists(resume_checkpoint):
        print(f"Resuming from checkpoint: {resume_checkpoint}")
        checkpoint = torch.load(resume_checkpoint, map_location=device)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        if scaler: scaler.load_state_dict(checkpoint['scaler_state'])
        
        global_step = checkpoint['global_step']
        start_epoch = checkpoint['epoch'] + 1
        criterion.phase = checkpoint['loss_phase']
        criterion.transition_start = checkpoint['loss_transition']
        criterion.val_lce_history = checkpoint['loss_lce_history']
        criterion.val_steps_history = checkpoint['loss_steps_history']

    steps_per_epoch = len(train_loader)
    
    # 5. Main Training Loop
    for epoch in range(start_epoch, start_epoch + epochs):
        model.train()
        running_loss = 0.0
        epoch_start = time.perf_counter()

        for batch_index, (images, masks, _) in enumerate(train_loader, start=1):
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if device.type == 'cuda':
                with torch.amp.autocast('cuda'):
                    seg_out = model(images)
                    loss = criterion(seg_out, masks, global_step)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                seg_out = model(images)
                loss = criterion(seg_out, masks, global_step)
                loss.backward()
                optimizer.step()

            scheduler.step()
            running_loss += loss.item()
            global_step += 1

            if batch_index % 100 == 0:
                print(f"Epoch [{epoch + 1}] | Step [{global_step}] | Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

        # 6. Evaluation Loop
        model.eval()
        val_lce_total = 0.0
        total_intersection = torch.zeros(num_seg_classes, device=device)
        total_union = torch.zeros(num_seg_classes, device=device)

        print(f"Starting evaluation for Epoch {epoch + 1}...")
        with torch.no_grad():
            for val_images, val_masks, _ in val_loader:
                val_images = val_images.to(device, non_blocking=True)
                val_masks = val_masks.to(device, non_blocking=True)
                
                if device.type == 'cuda':
                    with torch.amp.autocast('cuda'):
                        val_preds = model(val_images)
                        val_lce = criterion.ce(val_preds, val_masks)
                else:
                    val_preds = model(val_images)
                    val_lce = criterion.ce(val_preds, val_masks)

                val_lce_total += val_lce.item()
                
                # Fast vectorized Intersection over Union calculation
                preds_argmax = val_preds.argmax(dim=1)
                hist = torch.bincount(
                    num_seg_classes * val_masks.flatten() + preds_argmax.flatten(),
                    minlength=num_seg_classes ** 2
                ).reshape(num_seg_classes, num_seg_classes)
                
                intersection = torch.diag(hist)
                union = hist.sum(dim=1) + hist.sum(dim=0) - intersection
                
                total_intersection += intersection
                total_union += union

        # Calculate final metrics
        epoch_val_lce = val_lce_total / len(val_loader)
        
        # Avoid division by zero for classes absent in the validation set
        iou_per_class = total_intersection / (total_union + 1e-10)
        valid_classes_mask = total_union > 0
        epoch_val_miou = iou_per_class[valid_classes_mask].mean().item()

        # 7. Update DPW-Loss Triggers[cite: 7, 8]
        criterion.update_validation(epoch_val_lce, epoch_val_miou, global_step)

        print(f"--- Epoch [{epoch + 1}] Summary ---")
        print(f"Time: {(time.perf_counter() - epoch_start) / 60:.1f} min")
        print(f"Train Loss: {running_loss / steps_per_epoch:.4f}")
        print(f"Val CE Loss: {epoch_val_lce:.4f}")
        print(f"Val mIoU: {epoch_val_miou:.4f}")
        print(f"DPW-Loss Phase: {criterion.phase}\n")

        # 8. Save Checkpoint
        os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
        checkpoint_path = f"/kaggle/working/checkpoints/starnet_epoch_{epoch+1}.pth"
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'scaler_state': scaler.state_dict() if scaler else None,
            'loss_phase': criterion.phase,
            'loss_transition': criterion.transition_start,
            'loss_lce_history': criterion.val_lce_history,
            'loss_steps_history': criterion.val_steps_history
        }, checkpoint_path)
        print(f"Checkpoint saved to {checkpoint_path}")

    print("Training session complete.")

# Execute the replication pipeline
run_replication_training("/kaggle/input/models/tijesu26/starnet-epoch-15/pytorch/default/1/checkpoints/starnet_epoch_15.pth")

Found 118 filename classes across splits: ['train', 'val', 'test']


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

Resuming from checkpoint: /kaggle/input/models/tijesu26/starnet-epoch-15/pytorch/default/1/checkpoints/starnet_epoch_15.pth
Epoch [16] | Step [59470] | Loss: 0.4702 | LR: 5.00e-05
Epoch [16] | Step [59570] | Loss: 0.2032 | LR: 5.00e-05
Epoch [16] | Step [59670] | Loss: 0.5035 | LR: 5.00e-05
Epoch [16] | Step [59770] | Loss: 0.5110 | LR: 5.00e-05
Epoch [16] | Step [59870] | Loss: 0.2427 | LR: 5.00e-05
Epoch [16] | Step [59970] | Loss: 0.4364 | LR: 4.99e-05
Epoch [16] | Step [60070] | Loss: 0.4220 | LR: 4.99e-05
Epoch [16] | Step [60170] | Loss: 0.8160 | LR: 4.99e-05
Epoch [16] | Step [60270] | Loss: 0.2171 | LR: 4.99e-05
Epoch [16] | Step [60370] | Loss: 0.2883 | LR: 4.99e-05
Epoch [16] | Step [60470] | Loss: 0.5034 | LR: 4.99e-05
Epoch [16] | Step [60570] | Loss: 0.7196 | LR: 4.98e-05
Epoch [16] | Step [60670] | Loss: 0.3590 | LR: 4.98e-05
Epoch [16] | Step [60770] | Loss: 0.1751 | LR: 4.98e-05
Epoch [16] | Step [60870] | Loss: 0.3880 | LR: 4.98e-05
Epoch [16] | Step [60970] | Loss: 0.